# Batch Scraper — Trackers, Consent & CNIL Compliance Audit

This notebook replaces two earlier ones that ended up doing the same job twice:
`01_scraper_batch_tracker_consent` and `03_scraper_cnil_compliance_checks`
(both moved to `archive/`). 03 was really "01, done properly" — it duplicated
01's driver/tracker/policy code almost line for line, then replaced 01's naive
consent check (just look for the word "cookie" somewhere on the page) with a
real click-based test: find the actual button, click reject, remeasure
trackers and cookies before vs after. 03 then also bolted on a second,
shadow-DOM-aware button finder afterward, as a separate re-scan pass over the
sites its first, simpler finder had missed, and merged the two result sets
back together with pandas.

There's no reason to keep two notebooks, or two button finders, for that.
This one keeps only the final, most capable version of every function —
the shadow-DOM + iframe button finder is just *the* finder, used from the
start, so there's no gap left to re-scan and no merge step needed.

Runs on `clean_data/urls_new.csv` — the newer batch of sites that hasn't
been scraped yet (the original `urls.csv` batch is already in
`clean_data/scraped_results_clean.csv` / `raw_data/cnil_audit_results.csv`).

In [4]:
import json
import re
import time
import unicodedata
from urllib.parse import urlparse, urljoin

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

## 1. Driver management

Batch runs over thousands of sites WILL eventually crash a Chrome session.
`ensure_driver()` is called before every site in the loops below instead of
assuming one driver survives the whole run.

In [5]:
def create_driver():
    """Headless Chrome with network logging enabled (needed for tracker detection)."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(15)
    return driver


def is_driver_alive(driver):
    try:
        driver.title
        return True
    except Exception:
        return False


def ensure_driver(driver):
    """Returns a working driver, recreating it if the current session has crashed."""
    if driver is None or not is_driver_alive(driver):
        try:
            driver.quit()
        except Exception:
            pass
        return create_driver()
    return driver

## 2. Tracker detection

Loads the Disconnect.me verified tracker list once, and reads Chrome's
performance log to see which third-party domains actually got contacted.
The log buffer clears on every `get_log()` call — that's what makes a clean
before/after comparison possible later, in the differential test (Section 7).

In [6]:
with open("../raw_data/disconnect_services.json", "r") as f:
    disconnect_data = json.load(f)


def build_domain_lookup(disconnect_data):
    """Flattens the nested Disconnect.me JSON into {domain: category}."""
    domain_to_category = {}
    for category, services in disconnect_data["categories"].items():
        for service in services:
            for company_name, urls_dict in service.items():
                for homepage, domains in urls_dict.items():
                    for domain in domains:
                        domain_to_category[domain] = category
    return domain_to_category


domain_to_category = build_domain_lookup(disconnect_data)
print(f"Loaded {len(domain_to_category)} known tracker domains")


def is_known_ad_network(domain):
    return any(
        known_domain in domain and category in ("Advertising", "Analytics")
        for known_domain, category in domain_to_category.items()
    )


def extract_domains_from_logs(driver):
    """Reads every domain contacted since the last call to this function."""
    try:
        logs = driver.get_log("performance")
    except Exception:
        return set()

    domains = set()
    for entry in logs:
        try:
            message = json.loads(entry["message"])["message"]
            if message.get("method") == "Network.requestWillBeSent":
                domain = urlparse(message["params"]["request"]["url"]).netloc
                if domain:
                    domains.add(domain)
        except (KeyError, json.JSONDecodeError):
            continue
    return domains


def count_trackers_in_domains(domains, main_domain):
    """Given a set of contacted domains, returns (third_party_count, known_tracker_count)."""
    third_party = [d for d in domains if main_domain not in d]
    trackers = [d for d in third_party if is_known_ad_network(d)]
    return len(third_party), len(trackers)

Loaded 4443 known tracker domains


## 3. Cookie consent — keywords & known CMP names

Root words (not full phrases), matched with word boundaries against
accent-stripped text — `reject` alone catches gov.uk's "Reject additional
cookies", and stripping accents means `confidentialite`/`confidentialité` or
`odrzuc`/`odrzuć` are the same match instead of needing both spellings
hand-listed. `KNOWN_CMP_NAMES` grew through manual validation against
BigQuery ground truth (Setupad, Conversant and Funding Choices were all
found that way, in the original notebooks' debugging sections).

In [7]:
def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))


REJECT_BUTTON_KEYWORDS = [
    # English
    "reject", "reject all", "decline", "deny", "refuse",
    "do not accept", "do not agree", "don't accept", "i don't agree", "disagree",
    # German
    "ablehnen", "alle ablehnen", "ablehnen und schliessen",
    "nur erforderliche", "nur notwendige akzeptieren", "nicht einverstanden",
    "verweigern",
    # French
    "refuser", "tout refuser", "refuser tout", "je refuse",
    "continuer sans accepter", "n'accepter que les cookies essentiels",
    "refuser les cookies non essentiels",
    # Italian
    "rifiuta", "rifiuta tutto", "rifiuto", "nega", "negare",
    "non accetto", "continua senza accettare",
]

ACCEPT_BUTTON_KEYWORDS = [
    # English
    "accept", "accept all", "agree", "allow all", "i agree", "got it", "ok, got it",
    # German
    "akzeptieren", "alle akzeptieren", "zustimmen", "ich stimme zu",
    "einverstanden", "alle zulassen",
    # French
    "accepter", "tout accepter", "j'accepte", "autoriser tout", "j'accepte tout",
    # Italian
    "accetta", "accetta tutto", "accetto", "acconsento", "consenti tutto",
]

KNOWN_CMP_NAMES = [
    "didomi", "onetrust", "cookiebot", "usercentrics",
    "trustarc", "klaro", "osano", "cookieyes",
    "complianz", "iubenda", "quantcast",
    "funding choices", "fundingchoices",
    "setupad", "conversant", "cookie notice",
    "axeptio", "sourcepoint", "commanders act", "trustcommander",
]


def build_keyword_pattern(keywords):
    """One compiled regex matching any keyword as a whole word/phrase, case-insensitive,
    against accent-stripped text -- avoids false positives like 'accept' inside 'unacceptable'."""
    escaped = [re.escape(strip_accents(k)) for k in keywords]
    return re.compile(r"\b(" + "|".join(escaped) + r")\b", re.IGNORECASE)


REJECT_PATTERN = build_keyword_pattern(REJECT_BUTTON_KEYWORDS)
ACCEPT_PATTERN = build_keyword_pattern(ACCEPT_BUTTON_KEYWORDS)

## 4. Finding the actual clickable button (shadow DOM + iframes)

The one and only button finder. This is the shadow-DOM-piercing version from
the old notebook's late-stage "gap re-scan" — that pass only existed because
the *first*, simpler finder missed shadow-DOM banners and
`<input type="submit">` buttons (Amazon uses these for its reject button,
and they have no `.text`, only a `value` attribute). There's no reason to
run the weaker version first anymore, so this is used directly in the main
audit function in Section 7.

In [8]:
def safe_default_content(driver):
    """Switches to the main document without ever raising, even mid-crash."""
    try:
        driver.switch_to.default_content()
    except WebDriverException:
        pass


def get_element_label(el):
    """Best available text label for an element: .text for buttons, 'value' for
    <input type="submit">/<input type="button"> (which have no text nodes),
    aria-label as a last resort for icon-only buttons."""
    try:
        txt = el.text.strip()
        if txt:
            return txt.lower()
    except Exception:
        pass
    try:
        value = el.get_attribute("value")
        if value:
            return value.strip().lower()
    except Exception:
        pass
    try:
        aria = el.get_attribute("aria-label")
        if aria:
            return aria.strip().lower()
    except Exception:
        pass
    return ""


def find_buttons_deep(driver):
    """Shadow-DOM-piercing search in the current frame/context."""
    deep_search_js = """
    function deepQueryButtons(root) {
        let results = [];
        function walk(node) {
            if (!node) return;
            const matches = node.querySelectorAll(
                'button, a[role="button"], [role="button"], ' +
                'input[type="button"], input[type="submit"]'
            );
            for (const el of matches) { results.push(el); }
            const all = node.querySelectorAll('*');
            for (const el of all) {
                if (el.shadowRoot) { walk(el.shadowRoot); }
            }
        }
        walk(root);
        return results;
    }
    return deepQueryButtons(document.documentElement);
    """
    try:
        return driver.execute_script(deep_search_js)
    except WebDriverException:
        return []


def find_clickable_button(driver, pattern, max_iframe_depth=2):
    """Searches the main doc + shadow roots (any depth) + iframes (nested up to
    max_iframe_depth, each with their own shadow roots) for an element whose label
    matches `pattern`. Returns (element, size_dict) or (None, None). Leaves the
    driver switched into whichever frame the element was found in."""

    def search_here():
        for el in find_buttons_deep(driver):
            label = get_element_label(el)
            if label and pattern.search(strip_accents(label)):
                return el, el.size
        return None, None

    def recurse(depth):
        found, size = search_here()
        if found is not None:
            return found, size
        if depth >= max_iframe_depth:
            return None, None
        try:
            iframes = driver.find_elements("tag name", "iframe")
        except WebDriverException:
            return None, None
        for i in range(len(iframes)):
            try:
                current_iframes = driver.find_elements("tag name", "iframe")
                driver.switch_to.frame(current_iframes[i])
            except (WebDriverException, IndexError):
                continue
            result, size = recurse(depth + 1)
            if result is not None:
                return result, size
            try:
                driver.switch_to.parent_frame()
            except WebDriverException:
                pass
        return None, None

    try:
        driver.switch_to.default_content()
    except WebDriverException:
        return None, None
    result, size = recurse(depth=0)
    if result is None:
        safe_default_content(driver)
    return result, size


def click_element_safely(driver, element):
    try:
        driver.execute_script("arguments[0].click();", element)
        return True
    except Exception:
        try:
            element.click()
            return True
        except Exception:
            return False

## 5. Cookie classification

CNIL rule: essential/technical cookies may be set before consent; anything
else (analytics, advertising) should not be. A cookie counts as
non-essential if it's a known tracker domain, or genuinely third-party, or
first-party but its name doesn't look essential. Long-lived = over CNIL's
own 13-month limit.

In [9]:
ESSENTIAL_NAME_HINTS = {"cf_", "session", "phpsessid", "csrf", "__cf", "consent"}


def classify_cookies(cookies, main_domain):
    """Returns (total, non_essential_count, long_lived_count)."""
    non_essential = 0
    long_lived = 0

    for c in cookies:
        cookie_domain = c.get("domain", "").lstrip(".")
        name_lower = c.get("name", "").lower()

        is_first_party = main_domain in cookie_domain
        is_known_tracker_domain = is_known_ad_network(cookie_domain)
        looks_essential_by_name = any(hint in name_lower for hint in ESSENTIAL_NAME_HINTS)

        if is_known_tracker_domain or not is_first_party or (is_first_party and not looks_essential_by_name):
            non_essential += 1

        expiry = c.get("expiry")
        if expiry and (expiry - time.time()) / 86400 > 13 * 30:
            long_lived += 1

    return len(cookies), non_essential, long_lived

## 6. Privacy policy link

In [10]:
POLICY_LINK_WORDS = [
    # English
    "privacy policy", "privacy notice", "privacy statement",
    "data protection", "legal notice",
    # German
    "datenschutz", "datenschutzerklarung", "datenschutzerklärung",
    "datenschutzhinweise", "impressum",
    # French
    "politique de confidentialite", "politique de confidentialité",
    "mentions legales", "mentions légales",
    "protection des donnees", "protection des données",
    # Italian
    "informativa sulla privacy", "politica sulla privacy",
    "informativa privacy", "trattamento dei dati", "note legali",
    # Generic fallback (kept broad, catches the rest)
    "privacy", "cookies",
]


def find_policy_link(driver, page_url):
    """Looks for a privacy policy link on the current page. Returns the URL or None."""
    try:
        html = driver.page_source
    except Exception:
        return None

    soup = BeautifulSoup(html, "html.parser")
    matches = {}
    for link in soup.find_all("a", href=True):
        link_text = link.get_text().lower()
        for word in POLICY_LINK_WORDS:
            if word in link_text:
                matches[word] = urljoin(page_url, link["href"])

    for word in POLICY_LINK_WORDS:
        if word in matches:
            return matches[word]
    return None

## 7. The main audit function

One call per URL, used by the batch loop in Section 9. Wrapped in one outer
try/except: any Selenium call can time out unpredictably on a slow or
degraded page, and with thousands of sites in the loop, one bad site should
never take down the whole run — whatever was collected before the failure
is kept.

In [11]:
def audit_site(driver, url):
    """Full audit of one URL: trackers, cookies, CMP, consent buttons, the
    reject-click differential test, privacy policy, robots.txt."""
    row = {"url": url, "reachable": False, "error": None}
    main_domain = urlparse(url).netloc.replace("www.", "")

    try:
        driver.get(url)
        time.sleep(5)  # let the CMP + third-party scripts load
    except Exception as e:
        row["error"] = str(e)
        return row
    row["reachable"] = True

    try:
        # --- cookies + trackers BEFORE any interaction ---
        cookies_before = driver.get_cookies()
        total_before, non_essential_before, long_lived_before = classify_cookies(cookies_before, main_domain)
        row["cookies_before_interaction"] = total_before
        row["non_essential_cookies_before_interaction"] = non_essential_before
        row["long_lived_cookies_count"] = long_lived_before

        domains_before = extract_domains_from_logs(driver)
        third_party_before, trackers_before = count_trackers_in_domains(domains_before, main_domain)
        row["tracker_count"] = third_party_before
        row["ad_network_count"] = trackers_before

        # --- CMP name detection ---
        html_lower = driver.page_source.lower()
        row["cmp_detected"] = next((name for name in KNOWN_CMP_NAMES if name in html_lower), None)

        # --- TCF API presence ---
        try:
            row["tcf_api_present"] = bool(
                driver.execute_script("return typeof window.__tcfapi === 'function';")
            )
        except WebDriverException:
            row["tcf_api_present"] = False

        # --- find accept / reject buttons ---
        accept_el, accept_size = find_clickable_button(driver, ACCEPT_PATTERN)
        safe_default_content(driver)
        reject_el, reject_size = find_clickable_button(driver, REJECT_PATTERN)
        safe_default_content(driver)

        row["has_accept_button"] = accept_el is not None
        row["has_reject_button"] = reject_el is not None
        row["has_cookie_banner"] = row["has_accept_button"] or row["has_reject_button"]

        # --- CNIL requirement: is reject as visually prominent as accept? ---
        row["reject_as_visible_as_accept"] = None
        if accept_size and reject_size:
            a_area = accept_size.get("width", 0) * accept_size.get("height", 0)
            r_area = reject_size.get("width", 0) * reject_size.get("height", 0)
            if max(a_area, r_area) > 0:
                row["reject_as_visible_as_accept"] = (min(a_area, r_area) / max(a_area, r_area)) >= 0.7

        # --- CNIL requirement: no pre-ticked boxes ---
        try:
            safe_default_content(driver)
            checkboxes = driver.find_elements("css selector", "input[type='checkbox']")
            row["prechecked_boxes_detected"] = any(
                cb.is_selected() for cb in checkboxes if cb.is_displayed()
            )
        except WebDriverException:
            row["prechecked_boxes_detected"] = None
        safe_default_content(driver)

        # --- differential test: does clicking reject actually work? ---
        row["reject_click_attempted"] = False
        row["reject_click_succeeded"] = False
        row["tracker_count_after_reject"] = None
        row["non_essential_cookies_after_reject"] = None

        if reject_el is not None:
            row["reject_click_attempted"] = True
            reject_el_fresh, _ = find_clickable_button(driver, REJECT_PATTERN)
            if reject_el_fresh is not None:
                row["reject_click_succeeded"] = click_element_safely(driver, reject_el_fresh)
            safe_default_content(driver)
            time.sleep(3)

            cookies_after = driver.get_cookies()
            _, non_essential_after, _ = classify_cookies(cookies_after, main_domain)
            row["non_essential_cookies_after_reject"] = non_essential_after

            domains_after = extract_domains_from_logs(driver)
            _, trackers_after = count_trackers_in_domains(domains_after, main_domain)
            row["tracker_count_after_reject"] = trackers_after

        # --- privacy policy link ---
        safe_default_content(driver)
        row["has_privacy_policy"] = find_policy_link(driver, url) is not None

        # --- robots.txt ---
        try:
            driver.get(f"{urlparse(url).scheme}://{main_domain}/robots.txt")
            time.sleep(1)
            row["robots_txt_present"] = "user-agent" in driver.page_source.lower()
        except WebDriverException:
            row["robots_txt_present"] = None

    except Exception as e:
        row["error"] = f"partial failure: {str(e)[:200]}"

    return row

## 8. Smoke test on a few known sites

Always test on a handful of sites before committing to a multi-hour run.

In [9]:
driver = create_driver()

test_sites = ["https://www.lemonde.fr/", "https://www.wikipedia.org/"]

test_results = []
for site in test_sites:
    driver = ensure_driver(driver)
    print("Auditing:", site)
    test_results.append(audit_site(driver, site))

driver.quit()
pd.DataFrame(test_results)

Auditing: https://www.lemonde.fr/
Auditing: https://www.wikipedia.org/


,url,reachable,error,cookies_before_interaction,non_essential_cookies_before_interaction,long_lived_cookies_count,tracker_count,ad_network_count,cmp_detected,tcf_api_present,...,has_reject_button,has_cookie_banner,reject_as_visible_as_accept,prechecked_boxes_detected,reject_click_attempted,reject_click_succeeded,tracker_count_after_reject,non_essential_cookies_after_reject,has_privacy_policy,robots_txt_present
0,https://www.lemonde.fr/,True,None,8,8,6,4,4,None,True,...,False,True,None,False,False,False,None,None,True,True
1,https://www.wikipedia.org/,True,None,5,5,0,2,2,None,False,...,False,False,None,False,False,False,None,None,True,True


## 9. Full run on `urls_new.csv`

Loads the newer batch of sites (not yet scraped) and runs the full audit,
saving progress every 100 sites so a crash partway through a multi-hour run
doesn't lose everything.

In [16]:
df_urls = pd.read_csv("../clean_data/urls_new.csv")
url_list = df_urls["url"].tolist()
print(f"Total URLs to scrape: {len(url_list)}")

Total URLs to scrape: 5000


In [11]:
driver = create_driver()

all_results = []

for i, url in enumerate(url_list):
    driver = ensure_driver(driver)
    row = audit_site(driver, url)
    all_results.append(row)

    if (i + 1) % 100 == 0:
        print(f"Done {i+1} / {len(url_list)}")
        pd.DataFrame(all_results).to_csv("../raw_data/scraped_results_progress_new.csv", index=False)

driver.quit()

df_raw = pd.DataFrame(all_results)
df_raw.to_csv("../raw_data/scraped_results_new.csv", index=False)
print("Saved scraped_results_new.csv —", df_raw.shape)

Done 100 / 5000
Done 200 / 5000
Done 300 / 5000
Done 400 / 5000
Done 500 / 5000
Done 600 / 5000
Done 700 / 5000
Done 800 / 5000
Done 900 / 5000
Done 1000 / 5000
Done 1100 / 5000
Done 1200 / 5000
Done 1300 / 5000
Done 1400 / 5000
Done 1500 / 5000
Done 1600 / 5000
Done 1700 / 5000
Done 1800 / 5000
Done 1900 / 5000
Done 2000 / 5000
Done 2100 / 5000
Done 2200 / 5000
Done 2300 / 5000
Done 2400 / 5000
Done 2500 / 5000
Done 2600 / 5000
Done 2700 / 5000


KeyboardInterrupt: 

In [12]:
df_raw = pd.DataFrame(all_results)
df_raw.to_csv("../raw_data/scraped_results_new.csv", index=False)
print("Saved scraped_results_new.csv —", df_raw.shape)

Saved scraped_results_new.csv — (2755, 21)


In [13]:
import pandas as pd

df_progress = pd.read_csv("../raw_data/scraped_results_new.csv")
already_done = len(df_progress)
print(f'Already scraped: {already_done} rows')

Already scraped: 2755 rows


In [14]:
driver = create_driver()

all_results = list(df_progress.to_dict('records'))

remaining_urls = url_list[already_done:]
print(f'Remaining to scrape: {len(remaining_urls)}')

for i, url in enumerate(remaining_urls):
    driver = ensure_driver(driver)
    row = audit_site(driver, url)
    all_results.append(row)

    if (i + 1) % 100 == 0:
        print(f"Done {i+1} / {len(remaining_urls)} (total so far: {len(all_results)})")
        pd.DataFrame(all_results).to_csv("../raw_data/scraped_results_progress_new2.csv", index=False)

driver.quit()

df_raw = pd.DataFrame(all_results)
df_raw.to_csv("../raw_data/scraped_results_new2.csv", index=False)
print("Saved scraped_results_new.csv —", df_raw.shape)

Remaining to scrape: 2245
Done 100 / 2245 (total so far: 2855)
Done 200 / 2245 (total so far: 2955)
Done 300 / 2245 (total so far: 3055)
Done 400 / 2245 (total so far: 3155)
Done 500 / 2245 (total so far: 3255)
Done 600 / 2245 (total so far: 3355)
Done 700 / 2245 (total so far: 3455)
Done 800 / 2245 (total so far: 3555)
Done 900 / 2245 (total so far: 3655)
Done 1000 / 2245 (total so far: 3755)
Done 1100 / 2245 (total so far: 3855)
Done 1200 / 2245 (total so far: 3955)


KeyboardInterrupt: 

In [15]:
print(f"Done {i+1} / {len(remaining_urls)} (total so far: {len(all_results)})")
pd.DataFrame(all_results).to_csv("../raw_data/scraped_results_progress_new2.csv", index=False)

Done 1207 / 2245 (total so far: 3961)


In [16]:
df_raw = pd.DataFrame(all_results)
df_raw.to_csv("../raw_data/scraped_results_new2.csv", index=False)
print("Saved scraped_results_new.csv —", df_raw.shape)

Saved scraped_results_new.csv — (3961, 21)


In [ ]:
import pandas as pd

df_progress2 = pd.read_csv("../raw_data/scraped_results_new3.csv")
already_done2 = len(df_progress2)
print(f'Already scraped: {already_done2} rows')

Already scraped: 3961 rows


In [18]:
driver = create_driver()

all_results = list(df_progress2.to_dict('records'))

remaining_urls = url_list[already_done2:]
print(f'Remaining to scrape: {len(remaining_urls)}')

for i, url in enumerate(remaining_urls):
    driver = ensure_driver(driver)
    row = audit_site(driver, url)
    all_results.append(row)

    if (i + 1) % 100 == 0:
        print(f"Done {i+1} / {len(remaining_urls)} (total so far: {len(all_results)})")
        pd.DataFrame(all_results).to_csv("../raw_data/scraped_results_progress_new3.csv", index=False)

driver.quit()

df_raw = pd.DataFrame(all_results)
df_raw.to_csv("../raw_data/scraped_results_new3.csv", index=False)
print("Saved scraped_results_new.csv —", df_raw.shape)

Remaining to scrape: 1039
Done 100 / 1039 (total so far: 4061)


KeyboardInterrupt: 

In [12]:
print(f"Done {i+1} / {len(remaining_urls)} (total so far: {len(all_results)})")
pd.DataFrame(all_results).to_csv("../raw_data/scraped_results_progress_new3.csv", index=False)

NameError: name 'i' is not defined

In [13]:
df_raw = pd.DataFrame(all_results)
df_raw.to_csv("../raw_data/scraped_results_new3.csv", index=False)
print("Saved scraped_results_new.csv —", df_raw.shape)

NameError: name 'all_results' is not defined

In [17]:
import pandas as pd

df_progress3 = pd.read_csv("../raw_data/scraped_results_new3.csv")
already_done3 = len(df_progress3)
print(f'Already scraped: {already_done3} rows')

Already scraped: 4096 rows


In [18]:
driver = create_driver()

all_results = list(df_progress3.to_dict('records'))

remaining_urls = url_list[already_done3:]
print(f'Remaining to scrape: {len(remaining_urls)}')

for i, url in enumerate(remaining_urls):
    driver = ensure_driver(driver)
    row = audit_site(driver, url)
    all_results.append(row)

    if (i + 1) % 100 == 0:
        print(f"Done {i+1} / {len(remaining_urls)} (total so far: {len(all_results)})")
        pd.DataFrame(all_results).to_csv("../raw_data/scraped_results_progress_new4.csv", index=False)

driver.quit()

df_raw = pd.DataFrame(all_results)
df_raw.to_csv("../raw_data/scraped_results_new4.csv", index=False)
print("Saved scraped_results_new.csv —", df_raw.shape)

Remaining to scrape: 904
Done 100 / 904 (total so far: 4196)
Done 200 / 904 (total so far: 4296)
Done 300 / 904 (total so far: 4396)
Done 400 / 904 (total so far: 4496)
Done 500 / 904 (total so far: 4596)
Done 600 / 904 (total so far: 4696)
Done 700 / 904 (total so far: 4796)
Done 800 / 904 (total so far: 4896)
Done 900 / 904 (total so far: 4996)
Saved scraped_results_new.csv — (5000, 21)


In [19]:
driver.quit()

In [21]:

df1 = pd.read_csv("../raw_data/scraped_results_progress_new.csv")
df2 = pd.read_csv("../raw_data/scraped_results_progress_new2.csv")
df3 = pd.read_csv("../raw_data/scraped_results_progress_new3.csv")
df4= pd.read_csv("../raw_data/scraped_results_progress_new4.csv")

df = pd.concat([df1, df2, df3,df4], ignore_index=True)
df.to_csv("../raw_data/scraped_results_progress_new_combined.csv", index=False)

## remember to concatanated the results

## 10. Cleaning

Same cleanup as the old notebook's tail end: drop sites that never loaded,
fix dtypes, drop duplicate URLs, save the model-ready file.

In [23]:
df = pd.read_csv("../raw_data/scraped_results_progress_new_combined.csv")
print("Starting shape:", df.shape)

df = df[df["reachable"] == True].copy()
print("After dropping unreachable sites:", df.shape)

df = df.drop(columns=["error"])

bool_cols = [
    "has_accept_button", "has_reject_button", "has_cookie_banner",
    "reject_click_attempted", "reject_click_succeeded", "prechecked_boxes_detected",
    "tcf_api_present", "has_privacy_policy", "robots_txt_present",
]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

df["cmp_detected"] = df["cmp_detected"].fillna("none")

before = len(df)
df = df.drop_duplicates(subset="url")
print(f"Dropped {before - len(df)} duplicate URLs")

print(df.isnull().sum())

Starting shape: (17008, 21)
After dropping unreachable sites: (14136, 21)
Dropped 10035 duplicate URLs
url                                            0
reachable                                      0
cookies_before_interaction                     1
non_essential_cookies_before_interaction       1
long_lived_cookies_count                       1
tracker_count                                  1
ad_network_count                               1
cmp_detected                                   0
tcf_api_present                                1
has_accept_button                              1
has_reject_button                              1
has_cookie_banner                              1
reject_as_visible_as_accept                 2740
prechecked_boxes_detected                     13
reject_click_attempted                         1
reject_click_succeeded                         1
tracker_count_after_reject                  2521
non_essential_cookies_after_reject          2521
has_privacy_pol

In [24]:
df.head(10)

,url,reachable,cookies_before_interaction,non_essential_cookies_before_interaction,long_lived_cookies_count,tracker_count,ad_network_count,cmp_detected,tcf_api_present,has_accept_button,has_reject_button,has_cookie_banner,reject_as_visible_as_accept,prechecked_boxes_detected,reject_click_attempted,reject_click_succeeded,tracker_count_after_reject,non_essential_cookies_after_reject,has_privacy_policy,robots_txt_present
0,https://telekom.de,True,5.0,5.0,0.0,3.0,3.0,none,False,True,True,True,True,False,True,True,2.0,6.0,True,True
3,https://t-online.de,True,3.0,1.0,0.0,8.0,1.0,sourcepoint,True,True,False,True,NaN,False,False,False,NaN,NaN,True,True
4,https://bbc.co.uk,True,16.0,16.0,6.0,25.0,15.0,none,True,True,True,True,False,False,True,True,16.0,25.0,True,True
5,https://www.gov.uk,True,1.0,1.0,0.0,4.0,1.0,none,False,True,True,True,True,False,True,True,0.0,2.0,True,True
6,https://amazon.co.uk,True,10.0,10.0,0.0,8.0,4.0,cookie notice,True,True,True,True,True,False,True,True,0.0,11.0,True,True
8,https://dailymail.co.uk,True,15.0,15.0,4.0,30.0,15.0,trustarc,True,True,True,True,True,False,True,True,5.0,19.0,True,True
9,https://amazon.de,True,10.0,10.0,0.0,10.0,6.0,cookie notice,True,True,True,True,True,False,True,True,0.0,10.0,True,True
11,https://google.de,True,4.0,4.0,2.0,9.0,4.0,none,False,False,False,False,NaN,False,False,False,NaN,NaN,False,True
12,https://telegraph.co.uk,True,31.0,31.0,12.0,26.0,13.0,sourcepoint,True,True,True,True,True,False,True,True,1.0,33.0,False,True
13,https://free.fr,True,1.0,1.0,0.0,5.0,1.0,didomi,False,True,True,True,True,False,True,True,0.0,2.0,False,True


In [25]:
df.shape

(4101, 20)

In [26]:
df.to_csv("../clean_data/scraped_results_new_clean.csv", index=False)
print(f"Saved scraped_results_new_clean.csv — {len(df)} rows")

Saved scraped_results_new_clean.csv — 4101 rows


# EDA - Features selection

In [4]:

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_csv('../clean_data/scraped_results_new_clean.csv')
print('Shape:', df.shape)
df.head()

Shape: (4101, 20)


,url,reachable,cookies_before_interaction,non_essential_cookies_before_interaction,long_lived_cookies_count,tracker_count,ad_network_count,cmp_detected,tcf_api_present,has_accept_button,has_reject_button,has_cookie_banner,reject_as_visible_as_accept,prechecked_boxes_detected,reject_click_attempted,reject_click_succeeded,tracker_count_after_reject,non_essential_cookies_after_reject,has_privacy_policy,robots_txt_present
0,https://telekom.de,True,5.0,5.0,0.0,3.0,3.0,none,False,True,True,True,True,False,True,True,2.0,6.0,True,True
1,https://t-online.de,True,3.0,1.0,0.0,8.0,1.0,sourcepoint,True,True,False,True,NaN,False,False,False,NaN,NaN,True,True
2,https://bbc.co.uk,True,16.0,16.0,6.0,25.0,15.0,none,True,True,True,True,False,False,True,True,16.0,25.0,True,True
3,https://www.gov.uk,True,1.0,1.0,0.0,4.0,1.0,none,False,True,True,True,True,False,True,True,0.0,2.0,True,True
4,https://amazon.co.uk,True,10.0,10.0,0.0,8.0,4.0,cookie notice,True,True,True,True,True,False,True,True,0.0,11.0,True,True


In [5]:
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct.round(1))

reject_as_visible_as_accept                 66.8
non_essential_cookies_after_reject          61.5
tracker_count_after_reject                  61.5
robots_txt_present                           0.6
prechecked_boxes_detected                    0.3
has_privacy_policy                           0.1
has_cookie_banner                            0.0
reject_click_succeeded                       0.0
reject_click_attempted                       0.0
has_reject_button                            0.0
has_accept_button                            0.0
tcf_api_present                              0.0
ad_network_count                             0.0
tracker_count                                0.0
long_lived_cookies_count                     0.0
non_essential_cookies_before_interaction     0.0
cookies_before_interaction                   0.0
reachable                                    0.0
cmp_detected                                 0.0
url                                          0.0
dtype: float64


In [6]:
candidate_cols = [
    'tracker_count', 'ad_network_count', 'cookies_before_interaction',
    'non_essential_cookies_before_interaction', 'long_lived_cookies_count',
    'has_cookie_banner', 'has_accept_button', 'has_reject_button',
    'has_privacy_policy', 'robots_txt_present', 'tcf_api_present',
    'prechecked_boxes_detected'
]

for col in candidate_cols:
    print(f'--- {col} ---')
    if df[col].dtype in ('bool', 'object') or df[col].nunique() < 10:
        print(df[col].value_counts(dropna=False))
    else:
        print(df[col].describe())
    print()

--- tracker_count ---
count    4100.000000
mean       12.736585
std        11.737113
min         0.000000
25%         5.000000
50%         9.000000
75%        17.000000
max       148.000000
Name: tracker_count, dtype: float64

--- ad_network_count ---
count    4100.000000
mean        5.710488
std         7.366563
min         0.000000
25%         1.000000
50%         3.000000
75%         7.000000
max        91.000000
Name: ad_network_count, dtype: float64

--- cookies_before_interaction ---
count    4100.000000
mean        5.971220
std         6.997414
min         0.000000
25%         1.000000
50%         4.000000
75%         8.000000
max        79.000000
Name: cookies_before_interaction, dtype: float64

--- non_essential_cookies_before_interaction ---
count    4100.000000
mean        5.393902
std         6.716633
min         0.000000
25%         1.000000
50%         3.000000
75%         7.000000
max        77.000000
Name: non_essential_cookies_before_interaction, dtype: float64

--- lo

let's check how concentrated the real CMP provider names are -- decides whether to use the raw category or collapse into a simple has_known_cmp boolean.

In [7]:
print('Unique CMP values:', df['cmp_detected'].nunique())
print()
print(df['cmp_detected'].value_counts().head(15))
print()
pct_no_cmp = (df['cmp_detected'] == 'none').mean() * 100
print(f'% with no recognized CMP: {pct_no_cmp:.1f}%')

Unique CMP values: 21

cmp_detected
none              2403
onetrust           634
didomi             172
usercentrics       156
sourcepoint        144
cookiebot          133
trustarc           113
iubenda             62
cookie notice       56
cookieyes           40
osano               35
trustcommander      30
complianz           28
klaro               28
quantcast           22
Name: count, dtype: int64

% with no recognized CMP: 58.6%


Check the USABLE subset specifically --
this is the slice I would filter to for the differential-test analysis.

In [8]:
diff_cols = ['reject_click_attempted', 'reject_click_succeeded',
             'tracker_count_after_reject', 'non_essential_cookies_after_reject',
             'reject_as_visible_as_accept']

for col in diff_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts(dropna=False).head(5))
    print()

df_differential = df[df['reject_click_attempted'] == True].copy()
print(f'Usable differential-test subset: {len(df_differential)} rows ({len(df_differential)/len(df)*100:.1f}%)')

--- reject_click_attempted ---
reject_click_attempted
False    2517
True     1583
NaN         1
Name: count, dtype: int64

--- reject_click_succeeded ---
reject_click_succeeded
False    2528
True     1572
NaN         1
Name: count, dtype: int64

--- tracker_count_after_reject ---
tracker_count_after_reject
NaN    2521
0.0     684
1.0     340
2.0     164
3.0      63
Name: count, dtype: int64

--- non_essential_cookies_after_reject ---
non_essential_cookies_after_reject
NaN    2521
2.0     178
6.0     140
3.0     133
4.0     131
Name: count, dtype: int64

--- reject_as_visible_as_accept ---
reject_as_visible_as_accept
NaN      2740
True     1200
False     161
Name: count, dtype: int64

Usable differential-test subset: 1583 rows (38.6%)


The core CNIL finding -- compare tracker_count BEFORE vs AFTER reject,
only on the subset where the test was actually possible.

In [9]:
df_differential['tracker_reduction'] = (
    df_differential['tracker_count'] - df_differential['tracker_count_after_reject']
)

print('Average trackers BEFORE reject:', df_differential['tracker_count'].mean().round(2))
print('Average trackers AFTER reject:', df_differential['tracker_count_after_reject'].mean().round(2))
print('Average reduction:', df_differential['tracker_reduction'].mean().round(2))
print()
print('% of sites where reject brought trackers to EXACTLY zero:',
      (df_differential['tracker_count_after_reject'] == 0).mean() * 100, '%')
print('% of sites where reject did NOTHING (same count before/after):',
      (df_differential['tracker_reduction'] <= 0).mean() * 100, '%')

Average trackers BEFORE reject: 15.67
Average trackers AFTER reject: 3.1
Average reduction: 12.53

% of sites where reject brought trackers to EXACTLY zero: 43.20909665192672 %
% of sites where reject did NOTHING (same count before/after): 3.9797852179406195 %


In [11]:
print('=== TIER 1 - Core ML features (use for ALL 4101 rows) ===')
for c in candidate_cols:
    print(' -', c)
print(' - has_known_cmp  (derived: cmp_detected != "none")')
print()
print('=== TIER 2 - CNIL differential findings (separate analysis, n=%d) ===' % len(df_differential))
for c in diff_cols:
    print(' -', c)
print()
print('Next step: build risk_label from Tier 1 features, train on all 4101 rows.')

=== TIER 1 - Core ML features (use for ALL 4101 rows) ===
 - tracker_count
 - ad_network_count
 - cookies_before_interaction
 - non_essential_cookies_before_interaction
 - long_lived_cookies_count
 - has_cookie_banner
 - has_accept_button
 - has_reject_button
 - has_privacy_policy
 - robots_txt_present
 - tcf_api_present
 - prechecked_boxes_detected
 - has_known_cmp  (derived: cmp_detected != "none")

=== TIER 2 - CNIL differential findings (separate analysis, n=1583) ===
 - reject_click_attempted
 - reject_click_succeeded
 - tracker_count_after_reject
 - non_essential_cookies_after_reject
 - reject_as_visible_as_accept

Next step: build risk_label from Tier 1 features, train on all 4101 rows.
